# Speculative Decoding Engine — GPU Benchmark

**One-click notebook** to run the full benchmark on Google Colab / Kaggle GPU.

Recommended runtime: **GPU (T4 or A100)**

Expected runtime: ~15–25 minutes on T4, ~5–10 minutes on A100.

## Cell 1: Environment Setup

In [ ]:
# Check GPU availability
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU Info:', result.stdout.strip() if result.returncode == 0 else 'No CUDA GPU found')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}, VRAM: {props.total_memory / 1024**3:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate matplotlib numpy tqdm
print('Dependencies installed.')

In [ ]:
# Clone repository (if running from Colab/Kaggle without the repo)
import os
if not os.path.exists('speculative-decoding-engine'):
    !git clone https://github.com/YOUR_USERNAME/speculative-decoding-engine.git
    %cd speculative-decoding-engine
    print('Repository cloned.')
else:
    %cd speculative-decoding-engine
    print('Using existing repository.')

## Cell 2: (Optional) HuggingFace Login for Gated Models

In [ ]:
# Uncomment if models require authentication
# from huggingface_hub import login
# login()  # Will prompt for HF token

## Cell 3: Quick Unit Test (Optional Sanity Check)

In [ ]:
# Run a subset of the test suite to verify correctness before benchmarking
!python -m pytest tests/test_equivalence.py -v -k 'greedy and gamma1' --tb=short 2>&1 | tail -30

## Cell 4: Run Full Benchmark

In [ ]:
# Full benchmark: gamma sweep [2, 4, 6, 8], all 12 prompts, 256 tokens each
# Auto-detects hardware and selects appropriate model pair
!python benchmarks/run_benchmark.py \
    --max-new-tokens 256 \
    --gamma 2 4 6 8 \
    --warmup 2 \
    --output benchmarks/results.json

## Cell 5: Run Smoke Benchmark (Fast, for Testing)

In [ ]:
# Quick smoke test: 1 prompt, gamma=[2,4], 64 tokens
!python benchmarks/run_benchmark.py --smoke

## Cell 6: Generate Plots

In [ ]:
!python benchmarks/plot_results.py --input benchmarks/results.json --output-dir assets/

## Cell 7: Display Results

In [ ]:
import json
import IPython.display as ipd
from IPython.display import Image, display

# Load and display summary
with open('benchmarks/results.json') as f:
    results = json.load(f)

meta = results['meta']
print(f"Target model:  {meta['target_model']}")
print(f"Draft model:   {meta['draft_model']}")
print(f"Device:        {meta['device']}")
print(f"Max tokens:    {meta['max_new_tokens']}")
print(f"Gamma values:  {meta['gamma_values']}")
print()

# Summary table
spec = results['speculative']
baseline = results['baseline']
baseline_tps = sum(r['throughput_tps'] for r in baseline) / len(baseline) if baseline else 0
print(f"Baseline mean throughput: {baseline_tps:.1f} tokens/sec")
print()
print(f"{'γ':>4} | {'TPS':>8} | {'Speedup':>8} | {'α (mean)':>9} | {'TTFT (ms)':>10}")
print("-" * 55)
for gamma in meta['gamma_values']:
    g_res = [r for r in spec if r['gamma'] == gamma]
    if g_res:
        mean_tps = sum(r['throughput_tps'] for r in g_res) / len(g_res)
        mean_speedup = sum(r['speedup_ratio'] for r in g_res) / len(g_res)
        alphas = [r['mean_acceptance_rate'] for r in g_res if r['mean_acceptance_rate']]
        mean_alpha = sum(alphas) / len(alphas) if alphas else 0.0
        mean_ttft = sum(r['ttft_ms'] for r in g_res) / len(g_res)
        print(f"  {gamma:>2} | {mean_tps:>8.1f} | {mean_speedup:>7.2f}x | {mean_alpha:>9.3f} | {mean_ttft:>10.1f}")

In [ ]:
# Display charts
print("=== Speedup vs Gamma ===")
display(Image('assets/speedup_vs_gamma.png'))
print("\n=== Acceptance Rate by Domain ===")
display(Image('assets/acceptance_by_domain.png'))

## Cell 8: Run Full Pytest Suite

In [ ]:
!python -m pytest tests/test_equivalence.py -v --tb=short 2>&1